In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import shutil

In [3]:
from config import config

In [4]:
root = Path("../../..")

In [5]:
SOURCE_DIR = Path("D:/MSc/3. Spring 2026/CSE715/Project/MERGE_Bimodal_Balanced/lyrics")
DEST_DIR = root / config.LYRICS_DIR_EN
METADATA_FILE = root / config.METADATA_DIR / "metadata_en.csv"
FEATURES_DIR = root / config.FEATURES_DIR

In [6]:
SOURCE_DIR.exists(), METADATA_FILE.exists(), FEATURES_DIR.exists() # have to be True

(True, True, True)

In [7]:
DEST_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
all_feature_files = list(FEATURES_DIR.glob("*.npy"))
parent_ids = [f.stem.split('_')[0] for f in all_feature_files]
unique_song_ids = sorted(list(set(parent_ids)))

print(f"Total .npy files found: {len(all_feature_files)}")
print(f"Total unique songs extracted: {len(unique_song_ids)}")
print("First 10 unique IDs:", unique_song_ids[:10])

Total .npy files found: 2803
Total unique songs extracted: 158
First 10 unique IDs: ['', '-Bfa-aVa7MU', '-ph4mykFp9I', '8dLOs8LK4OQ', 'A025', 'A039', 'A048', 'A167', 'BWIW9aBNmlk', 'CKfhGvUPXkY']


In [9]:
df_meta = pd.read_csv(METADATA_FILE)
relevant_meta = df_meta[df_meta['audio_file_stem'].isin(unique_song_ids)]

In [10]:
len(relevant_meta["audio_file_stem"].unique())

128

In [15]:
relevant_meta.head()

,audio_file_stem,lyric_file_stem,quadrant,genres,label
14,A025,L075,Q2,"East Coast Rap,Rap,Hardcore Rap",1
21,A039,L089,Q1,"Disco,R&B,Swedish Pop/Rock,Euro-Pop,Contempora...",0
25,A048,L098,Q3,"Adult Alternative Pop/Rock,Alternative/Indie R...",3
69,A167,L167,Q1,"Uptown Soul,Pop-Soul,R&B,Soul",0
84,MT0000027422,MT0000027422,Q3,Pop/Rock,3


In [11]:
def find_lyrics_file(quadrant, stem):
    """
    Find the .txt files in "<SOURCE_DIR>/<quadrant>/<stem>.txt".
    """
    candidate = SOURCE_DIR / quadrant / f"{stem}.txt"
    return candidate if candidate.exists() else None

In [12]:
def transfer_lyrics_file(src, dst=DEST_DIR, copy_instead_of_move=True):
    """
    Creates destination folder if not already present. Copies or moves the .txt file from the source folder to the destination folder.
    """
    shutil.copy2(src, dst) if copy_instead_of_move else shutil.move(str(src), str(dst))

In [13]:
missing = []
found = 0
for _, row in relevant_meta.iterrows():
    stem = str(row["lyric_file_stem"]).strip()
    quadrant = str(row["quadrant"]).strip()
    try:
        src_file = find_lyrics_file(quadrant=quadrant, stem=stem)
        if src_file is None: 
            missing.append(stem)
            continue
        transfer_lyrics_file(src=src_file)
        found += 1
    except Exception as e:
        print(f"{e}: {stem}")

In [14]:
missing, found

([], 128)